In [62]:
import numpy as np
import pandas as pd
import os

cols = ["Rndrng_NPI", "Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
        "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Stdzd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

# same cleaning as data_preprocessing.ipynb
keep = ((df["Rndrng_Prvdr_Cntry"] == "US")
        & (df["Tot_Srvcs"] >= df["Tot_Benes"])
        & (df["Tot_Bene_Day_Srvcs"] <= df["Tot_Srvcs"])
        & (df["Avg_Mdcr_Alowd_Amt"] > 0))
df = df[keep].reset_index(drop=True)
df["row_id"] = np.arange(len(df))
len(df) 

9781161

## Columns creation!

In [63]:
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))  # payment level (avoid log(0) = -inf for $0 amounts)
df["log_benes"] = np.log(df["Tot_Benes"])   # volume
df["srvcs_per_bene"] = np.log(df["Tot_Srvcs"] / df["Tot_Benes"]) # intensity
df["markup"] = np.log(df["Avg_Sbmtd_Chrg"].clip(lower=1e-6) / df["Avg_Mdcr_Alowd_Amt"])  # charge vs allowed

feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]   

# Peer groupping

In [64]:
MIN = 100 #peer group minimum size
levels = [["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],   # level 1
          ["HCPCS_Cd", "Place_Of_Srvc"], # level 2 (fallback)
          ["HCPCS_Cd"]] #level 3 (fallback)

df["peer_level"] = 0 # 0 = unscored, 1 = level 1, 2 = level 2, 3 = level 3
df["peer_id"] = -1 # no group id yet
df["peer_n"] = 0 # group size
for i, c in enumerate(levels, start=1):
    size = df.groupby(c, observed=True)["row_id"].transform("size")
    gid = df.groupby(c, observed=True).ngroup() + i * 10_000_000   # unique group id for each level (10M apart to avoid collisions)
    take = (df["peer_level"] == 0) & (size >= MIN)
    df.loc[take, "peer_id"] = gid[take]
    df.loc[take, "peer_n"] = size[take]
    df.loc[take, "peer_level"] = i

(df["peer_level"].value_counts(normalize=True) * 100).round(2)  # 0 = unscored

peer_level
1    94.56
2     4.36
0     0.78
3     0.29
Name: proportion, dtype: float64

# Robust z scores for our 5 columns

In [65]:
s = df[df["peer_level"]>0]
med = s.groupby("peer_id")[feats].transform("median") # median of each feature within each peer group
dev = (s[feats] - med).abs() # absolute deviation from median
mad = dev.groupby(s["peer_id"]).transform("median")  # median absolute deviation
scale = (1.4826 * mad).where(mad > 0, 1.2533 * dev.groupby(s["peer_id"]).transform("mean"))
rz = (s[feats] - med) / scale.where(scale > 0) # robust z-score (NaN if scale = 0)

rz_cols = [f"{f}_rz" for f in feats]
df[rz_cols] = rz.set_axis(rz_cols, axis=1) # rename columns, rows align by index (unscored -> NaN)

In [66]:
import os
os.makedirs("data/processed", exist_ok=True)

model_input = df[["row_id", "Rndrng_NPI", "peer_level", "peer_id", "peer_n"] + rz_cols].copy()
model_input[rz_cols] = model_input[rz_cols].astype("float32")      
model_input.to_parquet("data/processed/model_input.parquet", index=False)

## Explorations

In [67]:
model_input.head()

,row_id,Rndrng_NPI,peer_level,peer_id,peer_n,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz
0,0,1003000126,1,10026929,1213,-2.119651,0.920317,0.000000,0.501598
1,1,1003000126,1,10026931,18024,-1.696201,2.022970,0.000000,0.083158
2,2,1003000126,1,10026933,32360,-0.020690,0.251892,-0.674491,0.964575
3,3,1003000126,1,10026935,5592,-18.361212,-1.035447,-0.522472,0.230335
4,4,1003000126,1,10026937,35610,-3.948478,-0.515971,-0.413015,2.400614


In [68]:
model_input[model_input["peer_level"] == 0]["Rndrng_NPI"].nunique()

38872

In [69]:
df[df["peer_level"] == 0]["Rndrng_Prvdr_Type"].value_counts().head(10)

Rndrng_Prvdr_Type
Clinical Laboratory           18550
Diagnostic Radiology           3634
Orthopedic Surgery             3319
Ambulatory Surgical Center     3317
Urology                        2420
Ophthalmology                  2350
Otolaryngology                 2288
Nurse Practitioner             2233
General Surgery                2197
Physician Assistant            2180
Name: count, dtype: int64

In [70]:
df['high_markup'] = df['markup_rz']>3.5

In [71]:
df[["high_markup"]].value_counts(normalize=True).mul(100).round(2)

high_markup
False          98.58
True            1.42
Name: proportion, dtype: float64

In [72]:
df.groupby("Place_Of_Srvc")["high_markup"].mean().mul(100).round(2)

C:\Users\talic\AppData\Local\Temp\ipykernel_4432\1830304588.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Place_Of_Srvc")["high_markup"].mean().mul(100).round(2)


Place_Of_Srvc
F    1.30
O    1.48
Name: high_markup, dtype: float64

In [73]:
df["chrg_to_alowd"] = df["Avg_Sbmtd_Chrg"] / df["Avg_Mdcr_Alowd_Amt"]
by_type = df.groupby("Rndrng_Prvdr_Type", observed = True).agg(
    rows = ("chrg_to_alowd", "size"),
   med_markup = ("chrg_to_alowd", "median")
)
by_type[by_type['rows']>10000].sort_values("med_markup", ascending=False).head(10)

,rows,med_markup
Rndrng_Prvdr_Type,,
Anesthesiology,203587,9.193096
Certified Registered Nurse Anesthetist (CRNA),115717,8.769750
Emergency Medicine,210968,5.852103
Independent Diagnostic Testing Facility (IDTF),31764,5.792212
Ambulatory Surgical Center,60937,4.728252
Diagnostic Radiology,1119939,4.355949
Interventional Radiology,59869,4.315429
Pathology,144616,4.151911
Pain Management,44627,4.074121


In [74]:
df["prov_high_rows"]= df.groupby("Rndrng_NPI")["high_markup"].transform("sum")

In [75]:
df[df["Rndrng_NPI"] == "1992998702"]

,Rndrng_NPI,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,Rndrng_Prvdr_Cntry,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,...,peer_level,peer_id,peer_n,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz,high_markup,chrg_to_alowd,prov_high_rows
9781074,1992998702,Centralized Flu,90480,O,US,144,152.0,152,41.838224,38.788487,...,1,10005288,9807,NaN,0.152956,-0.259734,554.258086,True,1.078625,2
9781075,1992998702,Centralized Flu,90653,O,US,214,214.0,214,83.490000,81.820000,...,1,10005289,9030,0.000000,1.434678,0.000000,-1.065809,False,1.020411,2
9781076,1992998702,Centralized Flu,90677,O,US,20,20.0,20,297.102000,291.161000,...,1,10005298,5684,0.803021,0.000000,0.000000,-1.223073,False,1.020405,2
9781077,1992998702,Centralized Flu,91320,O,US,71,73.0,73,151.483562,148.452603,...,1,10005306,7768,-0.043664,-0.358741,-0.674491,-1.388057,False,1.020417,2
9781078,1992998702,Centralized Flu,91322,O,US,69,74.0,74,159.311757,156.127838,...,1,10005307,4801,0.412278,-0.231368,0.359399,-0.675608,False,1.020393,2
9781079,1992998702,Centralized Flu,G0008,O,US,218,221.0,221,41.128597,30.945656,...,1,10005308,9617,0.000000,0.510017,1.072448,60.626469,True,1.329059,2
9781080,1992998702,Centralized Flu,G0009,O,US,20,20.0,20,31.248500,27.225000,...,1,10005309,5857,0.000000,0.000000,0.000000,3.057465,False,1.147787,2


In [76]:
results = df.groupby("Rndrng_NPI").agg(
    n_rows = ("high_markup", "size"),
    n_high = ("high_markup", "sum"),
    share_high = ("high_markup", "mean")
)
top = results[(results["n_rows"]>= 20) & (results["share_high"]>0.5)].sort_values("share_high", ascending=False)
types = df[["Rndrng_NPI", "Rndrng_Prvdr_Type"]].drop_duplicates()
top = top.merge(types, left_index=True, right_on="Rndrng_NPI")
top = top.reset_index(drop=True)

In [77]:
top

,n_rows,n_high,share_high,Rndrng_NPI,Rndrng_Prvdr_Type
0,20,18,0.900000,1609971852,Diagnostic Radiology
1,23,20,0.869565,1124242029,Vascular Surgery
2,22,19,0.863636,1851311096,General Surgery
3,36,31,0.861111,1508094541,Diagnostic Radiology
4,21,18,0.857143,1518132083,Diagnostic Radiology
...,...,...,...,...,...
228,53,27,0.509434,1104208982,Diagnostic Radiology
229,55,28,0.509091,1053350298,Diagnostic Radiology
230,55,28,0.509091,1942232657,Diagnostic Radiology
231,63,32,0.507937,1003852039,Diagnostic Radiology


In [78]:
eligible = results[results["n_rows"] >= 20].merge(types, left_index=True, right_on="Rndrng_NPI")
eligible["Rndrng_Prvdr_Type"].value_counts(normalize=True).head(5)   

Rndrng_Prvdr_Type
Diagnostic Radiology    0.220200
Cardiology              0.095590
Family Practice         0.089308
Internal Medicine       0.088281
Dermatology             0.041194
Name: proportion, dtype: float64

In [79]:
check = df[df["Rndrng_NPI"].isin(top["Rndrng_NPI"]) & df["high_markup"]]
check.groupby("Rndrng_Prvdr_Type")["Place_Of_Srvc"].value_counts(normalize=True).loc["Diagnostic Radiology"]

C:\Users\talic\AppData\Local\Temp\ipykernel_4432\2197345982.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  check.groupby("Rndrng_Prvdr_Type")["Place_Of_Srvc"].value_counts(normalize=True).loc["Diagnostic Radiology"]


Place_Of_Srvc
F    0.913657
O    0.086343
Name: proportion, dtype: float64

Markup is a noisy feature for some specialties (e.g. radiology), because of billing structure, not only provider behavior.
- Power BI: add a note for radiology: "markup flags may reflect billing structure".
- Models: if Isolation Forest also ranks many radiologists at the top, this explains why.

## Baseline model

In [80]:
z = df[rz_cols].clip(lower=0)   # high side only, NOT capped at 10 here
n_flags = (z > 3.5).sum(axis=1)

print(f"1+ feature > 3.5: {(n_flags >= 1).mean():.2%}")
print(f"2+ features > 3.5: {(n_flags >= 2).mean():.2%}")
print(f"one > 10 OR 2+ > 3.5: {((z.max(axis=1) > 10) | (n_flags >= 2)).mean():.2%}")

1+ feature > 3.5: 5.79%
2+ features > 3.5: 0.15%
one > 10 OR 2+ > 3.5: 1.17%


In [81]:
high = df[rz_cols].clip(lower=0)  # only above peers
scored = df["peer_level"] > 0

z_sorted = np.sort(high.fillna(0).to_numpy(), axis=1) # each row sorted ascending
top1, top2 = z_sorted[:, -1], z_sorted[:, -2]# largest and 2nd largest z

# anomaly: one feature > 10 OR two+ features > 3.5  <=>  baseline_raw > 3.5
df["baseline_raw"] = pd.Series(np.maximum(top2, top1 * 3.5 / 10), index=df.index).where(scored)

df["n_flags"] = (high > 3.5).sum(axis=1)
df["flag"] = df["baseline_raw"] > 3.5
df["score"] = (df["baseline_raw"].rank(pct=True) * 100).round(1)
df["risk_tier"] = "None"
fl = df["flag"]
df.loc[fl, "risk_tier"] = pd.cut(df.loc[fl, "baseline_raw"].rank(pct=True),
                                 [0, 0.5, 0.85, 1], labels=["Low", "Medium", "High"]).astype(str)
names = [c.replace("_rz", "") for c in rz_cols]
mask = (high > 3.5).to_numpy()
df["reason"] = [", ".join(n for n, m in zip(names, row) if m) for row in mask]

print(f"flagged: {df['flag'].mean():.2%}")                     
print(pd.crosstab(df["risk_tier"], df["flag"]))

flagged: 1.17%
flag         False  True 
risk_tier                
High             0  17215
Low              0  57382
Medium           0  40168
None       9666396      0


In [82]:
out = df[["row_id", "score", "flag", "reason", "risk_tier"]].copy()
out.insert(1, "method", "baseline_zscore")
out.to_parquet("data/processed/baseline_output.parquet", index=False)
out.head()

,row_id,method,score,flag,reason,risk_tier
0,0,baseline_zscore,53.8,False,,None
1,1,baseline_zscore,69.2,False,,None
2,2,baseline_zscore,39.0,False,,None
3,3,baseline_zscore,14.2,False,,None
4,4,baseline_zscore,76.3,False,,None


In [83]:
out.sort_values("score", ascending=False).head(10)

,row_id,method,score,flag,reason,risk_tier
4757891,4757891,baseline_zscore,100.0,True,markup,High
4757892,4757892,baseline_zscore,100.0,True,markup,High
7388856,7388856,baseline_zscore,100.0,True,markup,High
2045369,2045369,baseline_zscore,100.0,True,"srvcs_per_bene, markup",High
1212899,1212899,baseline_zscore,100.0,True,"srvcs_per_bene, markup",High
4453094,4453094,baseline_zscore,100.0,True,markup,High
8175159,8175159,baseline_zscore,100.0,True,markup,High
4026457,4026457,baseline_zscore,100.0,True,markup,High
7390629,7390629,baseline_zscore,100.0,True,markup,High
2046255,2046255,baseline_zscore,100.0,True,markup,High


In [84]:
df.sort_values("baseline_raw", ascending=False)[["row_id", "baseline_raw", "reason"] + rz_cols].head(10)

,row_id,baseline_raw,reason,log_pay_rz,log_benes_rz,srvcs_per_bene_rz,markup_rz
6326823,6326823,2.047549e+10,markup,0.0,-0.774811,-1.254000,5.850140e+10
640831,640831,1.748912e+10,markup,0.0,0.471130,2.618855,4.996891e+10
1703718,1703718,1.665401e+10,markup,0.0,1.163443,-0.423368,4.758287e+10
7919249,7919249,1.665401e+10,markup,0.0,0.021483,0.253557,4.758287e+10
3389394,3389394,1.665401e+10,markup,0.0,1.127883,2.253458,4.758287e+10
1714115,1714115,1.665401e+10,markup,0.0,1.258588,1.089370,4.758287e+10
2698091,2698091,1.665401e+10,markup,0.0,0.529332,-0.369106,4.758287e+10
8258214,8258214,1.665401e+10,markup,0.0,1.508386,0.545367,4.758287e+10
2491600,2491600,1.650667e+10,markup,0.0,0.357469,0.766171,4.716190e+10
1612516,1612516,1.618912e+10,markup,0.0,1.433740,-0.252622,4.625463e+10


In [85]:
# within-group correlation
s = df[df["peer_level"] > 0].copy()
s["log_chrg"] = np.log(s["Avg_Sbmtd_Chrg"].clip(lower=1e-6))
g = s.groupby("peer_id")
for c in ["log_chrg", "markup"]:
    s[c + "_w"] = s[c] - g[c].transform("median")      # deviation from own group median
print(s[["log_chrg_w", "markup_w"]].corr(method="spearman"))

            log_chrg_w  markup_w
log_chrg_w    1.000000  0.928357
markup_w      0.928357  1.000000


## Compararion of 30 vs. 100 min group sizes

In [86]:
f = "log_pay"
s = df[scored]

# group-level summary statistics
grp = s.groupby("peer_id").agg(
    n=(f, "size"),
    level=("peer_level", "first"),
    median=(f, "median"),
    mean=(f, "mean"),
    std=(f, "std"),
    q25=(f, lambda x: x.quantile(0.25)),
    q75=(f, lambda x: x.quantile(0.75)),
    min=(f, "min"),
    max=(f, "max"),
    flag_rate=("flag", "mean"),
)
grp["iqr"] = grp["q75"] - grp["q25"]

In [87]:
# summary statistics for peer groups of sizes 30-99
small = grp[(grp["n"] >= 30) & (grp["n"] < 100)]
small.describe().round(3)

,n,level,median,mean,std,q25,q75,min,max,flag_rate,iqr
count,912.000,912.000,912.000,912.000,912.000,912.000,912.000,912.000,912.000,912.000,912.000
mean,60.539,2.285,4.096,4.067,0.256,3.930,4.223,3.426,4.452,0.033,0.293
std,20.229,0.452,1.443,1.409,0.303,1.385,1.498,1.476,1.545,0.067,0.525
min,30.000,2.000,-4.677,-4.684,0.000,-4.836,-4.613,-7.896,-4.104,0.000,0.000
25%,42.000,2.000,3.298,3.269,0.069,3.196,3.354,2.776,3.501,0.000,0.048
50%,59.000,2.000,4.155,4.136,0.156,3.994,4.255,3.586,4.409,0.011,0.126
75%,76.000,3.000,4.917,4.903,0.309,4.702,5.078,4.227,5.381,0.032,0.269
max,99.000,3.000,9.853,8.664,2.136,8.784,9.883,8.319,9.900,0.528,4.451


In [88]:
# summary statistics for peer groups of sizes 100+
big = grp[(grp["n"] >= 100)]
big.describe().round(3)

,n,level,median,mean,std,q25,q75,min,max,flag_rate,iqr
count,8609.000,8609.000,8609.000,8609.000,8609.000,8609.000,8609.000,8609.000,8609.000,8609.000,8609.000
mean,1119.578,1.198,3.686,3.654,0.163,3.588,3.753,2.734,3.893,0.015,0.165
std,3465.540,0.417,1.560,1.540,0.227,1.536,1.575,2.122,1.612,0.041,0.367
min,100.000,1.000,-13.816,-8.449,0.000,-13.816,-4.615,-13.816,-4.310,0.000,0.000
25%,154.000,1.000,2.849,2.811,0.039,2.764,2.903,2.044,2.991,0.000,0.015
50%,285.000,1.000,3.845,3.813,0.097,3.735,3.896,3.016,4.030,0.003,0.077
75%,716.000,1.000,4.604,4.580,0.204,4.505,4.666,3.903,4.838,0.015,0.151
max,91326.000,3.000,9.894,9.894,7.861,9.894,9.894,9.713,9.940,0.500,15.253


In [89]:
# Proportion of flagged claims in groups of size 100+
s.groupby(s["peer_n"] >= 100)["flag"].mean() * 100

peer_n
True    1.1826
Name: flag, dtype: float64

## Baseline model as a function: comparing results across min group sizes

In [90]:
LEVELS = [
    ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],  # level 1
    ["HCPCS_Cd", "Place_Of_Srvc"],  # level 2
    ["HCPCS_Cd"],   # level 3
]
feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]
rz_cols = [f + "_rz" for f in feats]

# baseline model function (rewritten code from above to make it a function)
def run_baseline(df, min_size):
    out = df[["row_id"] + LEVELS[0] + feats].copy()
    out["peer_level"] = 0
    out["peer_id"] = -1
    out["peer_n"] = 0

    # 1. Peer groups 
    for i, keys in enumerate(LEVELS, start=1):
        free = out[out["peer_level"] == 0]
        size = free.groupby(keys, observed=True)["row_id"].transform("size")
        take = size.index[size >= min_size]
        gid = free.loc[take].groupby(keys, observed=True).ngroup()
        out.loc[take, "peer_level"] = i
        out.loc[take, "peer_id"] = gid + i * 10_000_000
        out.loc[take, "peer_n"] = size.loc[take]

    # 2. Robust z inside the groups
    s = out[out["peer_level"] > 0]
    med = s.groupby("peer_id")[feats].transform("median")
    dev = (s[feats] - med).abs()
    mad = dev.groupby(s["peer_id"]).transform("median")
    scale = (1.4826 * mad).where(mad > 0,
                                 1.2533 * dev.groupby(s["peer_id"]).transform("mean"))
    rz = (s[feats] - med) / scale.where(scale > 0)

    out[rz_cols] = np.nan
    out.loc[s.index, rz_cols] = rz.to_numpy()

    # 3. Combined rule: 1 features > 10 OR 2+ features > 3.5 (we are not looking ar the values lower)
    high = out[rz_cols].clip(lower=0).fillna(0).to_numpy()
    z_sorted = np.sort(high, axis=1)
    raw = np.maximum(z_sorted[:, -2], z_sorted[:, -1] * 3.5 / 10)
    out["baseline_raw"] = pd.Series(raw, index=out.index).where(out["peer_level"] > 0)
    out["flag"] = out["baseline_raw"] > 3.5
       

    return out[["row_id", "Rndrng_Prvdr_Type", "peer_level", "peer_id",
                "peer_n", "baseline_raw", "flag"] + rz_cols]

In [91]:
# run on a 30 and 100 min groups 
b30  = run_baseline(df, 30)
b100 = run_baseline(df, 100)
# Rows that change peers when min size goes from 30 to 100:
moved = (b30["peer_level"] == 1) & (b100["peer_level"] == 2)
print(moved.sum(), b30.loc[moved, "flag"].mean()*100, b100.loc[moved, "flag"].mean()*100)

243137 1.9474617191130927 1.8088567350917386


In [92]:
# Flag Rates
print(b30["flag"].mean()*100, b100["flag"].mean()*100)

1.1774062404248329 1.1591670968303251


In [93]:
# How rows change peer level between min 30 and min 100
display(pd.crosstab(b30["peer_level"], b100["peer_level"],
                    rownames=["Level (min 30)"], colnames=["Level (min 100)"]))

Level (min 100),0,1,2,3
Level (min 30),,,,
0,38704,0,3243,1888
1,44936,9249171,243137,12073
2,38352,0,134229,7954
3,5804,0,871,799


In [94]:
# Do both thresholds flag the same rows? (all data)
display(pd.crosstab(b30["flag"], b100["flag"],
                    rownames=["Flag (min 30)"], colnames=["Flag (min 100)"]))


Flag (min 100),False,True
Flag (min 30),,
False,9660811,5186
True,6970,108194


In [95]:
# Same check, only rows that moved from level 1 to level 2
display(pd.crosstab(b30.loc[moved, "flag"], b100.loc[moved, "flag"],
                    rownames=["Flag (min 30)"], colnames=["Flag (min 100)"]))

Flag (min 100),False,True
Flag (min 30),,
False,235777,2625
True,2962,1773


In [96]:
f30  = b30.loc[moved, "flag"]
f100 = b100.loc[moved, "flag"]

both  = (f30 & f100).sum()
union = (f30 | f100).sum()
print(f"Flagged under both: {both:,}")
print(f"Flagged under at least one: {union:,}")

Flagged under both: 1,773
Flagged under at least one: 7,360


Among moved rows, 4,735 are flagged at min 30 and 4,398 at min 100, but only 1,773 are flagged under both. 
The flag rate is the same, but more than half of the flags depend on the threshold.

In [98]:

only100 = moved & ~b30["flag"] & b100["flag"]
only30  = moved & b30["flag"] & ~b100["flag"]

In [99]:
# Flagged under one threshold but not the other
t = "Rndrng_Prvdr_Type"
side = pd.concat([
    b100.loc[only100, t].value_counts().rename("Only at min 100"),
    b30.loc[only30, t].value_counts().rename("Only at min 30"),
], axis=1).fillna(0).astype(int)

side.sort_values("Only at min 100", ascending=False).head(10)

,Only at min 100,Only at min 30
Rndrng_Prvdr_Type,,
Ambulatory Surgical Center,815,85
Nurse Practitioner,149,267
Family Practice,112,91
Portable X-Ray Supplier,89,22
Hand Surgery,86,9
Internal Medicine,81,165
Hematology-Oncology,77,78
Clinical Laboratory,72,97
Cardiology,63,99


In [100]:
asc = only100 & (b100["Rndrng_Prvdr_Type"] == "Ambulatory Surgical Center")
(b100.loc[asc, rz_cols] > 3.5).mean().round(2)

log_pay_rz           0.97
log_benes_rz         0.04
srvcs_per_bene_rz    0.06
markup_rz            0.02
dtype: float64

**Findings:**
Flags that appear only at min 100 relates to the provider types payment structure: Ambulatory Surgical Centers (815 vs 85), Hand Surgery (86 vs 9), Interventional Pain Management (62 vs 3) and Portable X-Ray (89 vs 22). With min 100 these providers fall back into mixed groups and get compared with physicians, so they look abnormal even when they aren't. Min 30 keeps them compared to their own.

In [101]:
b100 = run_baseline(df, 100)

display(b100["peer_level"].value_counts(normalize=True).sort_index() * 100)
print(f"Flag rate: {b100['flag'].mean()*100:.2f}%")

peer_level
0     1.306552
1    94.561075
2     3.900151
3     0.232222
Name: proportion, dtype: float64

Flag rate: 1.16%


In [102]:
f = df[df["flag"]]

print(f"Flagged rows: {len(f):,} ({df['flag'].mean()*100:.2f}%)")
display(f["reason"].value_counts(normalize=True).mul(100).round(1).rename("% of flags"))
display(f["risk_tier"].value_counts().rename("rows"))

Flagged rows: 114,765 (1.17%)


reason
srvcs_per_bene                                44.6
markup                                        30.1
log_pay                                       12.2
srvcs_per_bene, markup                         4.8
log_benes, srvcs_per_bene                      4.4
log_pay, srvcs_per_bene                        2.0
log_benes, markup                              0.6
log_pay, markup                                0.6
log_pay, log_benes                             0.4
log_benes                                      0.1
log_pay, log_benes, srvcs_per_bene             0.1
log_benes, srvcs_per_bene, markup              0.1
log_pay, srvcs_per_bene, markup                0.0
log_pay, log_benes, markup                     0.0
log_pay, log_benes, srvcs_per_bene, markup     0.0
Name: % of flags, dtype: float64

risk_tier
Low       57382
Medium    40168
High      17215
Name: rows, dtype: int64

In [103]:
big = df[(df["HCPCS_Cd"] == "99213") & (df["peer_level"] == 1)]
pid = big.loc[big["flag"], "peer_id"].value_counts().index[0]   # группа 99213 с наибольшим числом флагов
g = df[df["peer_id"] == pid].copy()

print(g[["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"]].iloc[0].to_dict(), "| rows:", len(g))

g["srv_per_bene"] = g["Tot_Srvcs"] / g["Tot_Benes"]
g["charge_ratio"] = g["Avg_Sbmtd_Chrg"] / g["Avg_Mdcr_Alowd_Amt"]
show = ["Avg_Mdcr_Stdzd_Amt", "Tot_Benes", "srv_per_bene", "charge_ratio"]

top = g[g["flag"]].sort_values("score", ascending=False).head(5)
table = pd.concat([
    g[show].median().to_frame("Peer median").T,
    top.set_index("row_id")[show + ["score", "risk_tier", "reason"]],
])
table.round(2)

{'Rndrng_Prvdr_Type': 'Nurse Practitioner', 'HCPCS_Cd': '99213', 'Place_Of_Srvc': 'O'} | rows: 82322


,Avg_Mdcr_Stdzd_Amt,Tot_Benes,srv_per_bene,charge_ratio,score,risk_tier,reason
Peer median,51.76,45.0,1.15,2.43,NaN,NaN,NaN
2427709,58.14,15.0,32.33,1.18,99.5,Medium,srvcs_per_bene
9603219,60.56,59.0,42.66,1.19,99.5,Medium,srvcs_per_bene
381986,59.21,135.0,15.81,1.59,99.4,Low,srvcs_per_bene
1259552,59.23,21.0,17.90,2.47,99.4,Low,srvcs_per_bene
7226606,57.17,53.0,3.13,16.99,99.4,Low,"srvcs_per_bene, markup"


In [104]:
r = f["reason"].where(~f["reason"].str.contains(","), "2+ features")
r.value_counts(normalize=True).mul(100).round(1)

reason
srvcs_per_bene    44.6
markup            30.1
2+ features       12.9
log_pay           12.2
log_benes          0.1
Name: proportion, dtype: float64

In [105]:
nice = table.rename(columns={
    "Avg_Mdcr_Stdzd_Amt": "Std. payment ($)",
    "Tot_Benes": "Patients",
    "srv_per_bene": "Visits / patient",
    "charge_ratio": "Charge / allowed",
    "risk_tier": "Risk tier",
    "reason": "Reason",
}).drop(columns="score").fillna("")
nice

,Std. payment ($),Patients,Visits / patient,Charge / allowed,Risk tier,Reason
Peer median,51.764281,45.0,1.153846,2.433231,,
2427709,58.135340,15.0,32.333333,1.181013,Medium,srvcs_per_bene
9603219,60.559213,59.0,42.661017,1.185823,Medium,srvcs_per_bene
381986,59.214948,135.0,15.807407,1.591852,Low,srvcs_per_bene
1259552,59.226330,21.0,17.904762,2.470319,Low,srvcs_per_bene
7226606,57.171807,53.0,3.132075,16.991532,Low,"srvcs_per_bene, markup"
